# 시계열: 심전도 심박 예측

지난 주는 복강경 장면의 픽셀이었다. 이번 장은 한 번의 심박 파형이다. PhysioNet BIDMC 울혈성 심부전 기록에서 자른 길이 140의 심전도에서 앞 100샘플을 보고 뒤 40샘플을 LSTM과 ModernTCN이 예측한다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 를 먼저 고른다.

zip에는 심박 배열과 학습해 둔 LSTM·ModernTCN 가중치가 있다. 학습·추론 코드는 이 노트북 셀과 같은 폴더의 `.py`에 있다.
`/content`는 세션이 끝나면 비므로 zip은 드라이브에 남겨 둔다.


## 1. 데이터 zip

드라이브에서 해당 zip을 `링크가 있는 모든 사용자`로 공유한 뒤, 링크를 `ZIP_URL`에 넣는다.

파일이 100MB보다 크면 `wget`이 구글 확인 페이지를 받는다. 드라이브 링크는 `gdown`, 직접 열리는 주소는 `wget`을 쓴다.


In [ ]:
ZIP_URL = "https://drive.google.com/file/d/1hMYbwoa5cBphd3vAqWPeuBU5w_B405Bw/view?usp=sharing"

TASK = "timeseries"
ZIP_PATH = "/content/06_timeseries.zip"
DST = f"/content/{TASK}"


## 2. 패키지


In [ ]:
!pip install -q gdown


In [ ]:
!apt-get -qq update
!apt-get -qq install -y fonts-noto-cjk fonts-nanum
from pathlib import Path
from PIL import ImageFont


def hangul_font(size=14):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            font = ImageFont.truetype(path, size, **kw)
            print("hangul_font", path)
            return font
        except OSError:
            continue
    print("hangul_font missing")
    return None


hangul_font()


## 3. zip 받기


In [ ]:
from pathlib import Path
import os


def fetch_zip(url, dest):
    url = (url or "").strip()
    if not url or "여기에" in url:
        raise SystemExit(
            "위 셀 ZIP_URL에 구글 드라이브 공유 링크(또는 직접 zip 주소)를 넣는다."
        )

    is_drive = "drive.google.com" in url or "docs.google.com" in url

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000:
        print("이미 받은 파일이 있다:", dest, dest.stat().st_size, "bytes")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)

    os.environ["ZIP_URL"] = url
    os.environ["ZIP_PATH"] = str(dest)
    if is_drive:
        !gdown --fuzzy "$ZIP_URL" -O "$ZIP_PATH"
    else:
        !wget -c -O "$ZIP_PATH" "$ZIP_URL"

    print("받음", dest, dest.stat().st_size, "bytes")


fetch_zip(ZIP_URL, ZIP_PATH)


In [ ]:
!rm -rf {DST}


In [ ]:
!unzip -q -o {ZIP_PATH} -d /content


In [ ]:
from pathlib import Path
import torch
from IPython.display import display
from PIL import Image

ROOT = Path(DST)
if not ROOT.exists():
    raise SystemExit(f"{ROOT} 가 없다. zip 안에 과제 폴더가 있는지 확인한다.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))


def show_img(img, width=900):
    """셀 결과에 PIL 이미지를 띄운다. 너무 가로로 긴 장만 줄인다."""
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


## 4. 데이터가 어떻게 생겼는가

생체신호는 시간이 축인 1차원 파형이다. 이 장은 UCR ECG5000이다. 원 기록은 PhysioNet BIDMC 울혈성 심부전 데이터베이스의 chf07이고 심박을 잘라 길이 140으로 보간한 표다.

과제는 분류가 아니라 예측이다. 앞 100샘플을 보고 뒤 40샘플의 진폭을 맞춘다. 원 라벨 5(미분류)는 24개뿐이라 뺀다. 환자 한 명의 심박 5,000개라서 사람 사이 일반화는 이 장이 아니다. 심박은 서로 다른 샘플이라 같은 입력 `(B, 100, 1)`으로 LSTM과 ModernTCN을 비교한다.

아래는 입력·정답 모양, 분할에 들어 있는 심박 종류, 테스트에서 고른 심박이다. 학습은 공식 train만 쓰고 검증 400개는 공식 test에서 층화추출한다.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.insert(0, str(ROOT))
from classes import HORIZON, LOOKBACK, SEQ_LEN
from data import load_bundle

bundle = load_bundle(ROOT / "dataset")
print("한 심박", SEQ_LEN, "입력", LOOKBACK, "예측", HORIZON)
print("분할", bundle["meta"]["splits"])
show_img(Image.open(ROOT / "examples" / "window_schema.png"), width=980)
print()
show_img(Image.open(ROOT / "examples" / "counts.png"), width=980)
print()
show_img(Image.open(ROOT / "examples" / "gallery.png"), width=980)


## 5. 입력과 정답

한 샘플은 심박 하나다.

- 입력 `x`: 앞 100샘플  → 모양 `(100, 1)`
- 정답 `y`: 뒤 40샘플   → 모양 `(40, 1)`
- 배치로 묶으면 입력 `(B, 100, 1)`, 출력 `(B, 40, 1)`

학습 평균·표준편차는 train 파형으로만 구한다. 아래는 테스트에서 고른 네 심박의 입력 끝 4샘플과 정답 앞 4샘플이다.


In [ ]:
test = bundle["splits"]["test"]
print("test input", test["x"].shape, "label", test["y"].shape)
print()
for ex in bundle["meta"]["examples"]:
    i = ex["split_index"]
    x = test["x"][i, :, 0]
    y = test["y"][i, :, 0]
    print(f"{ex['id']} {ex['title']}")
    print("  input shape", tuple(test["x"][i].shape), "  끝 4샘플", np.round(x[-4:], 3))
    print("  output shape", tuple(test["y"][i].shape), "  앞 4샘플", np.round(y[:4], 3))


## 6. 모델

두 모델의 입출력 모양을 같게 둔다. 그래야 MAE와 RMSE를 나란히 적을 수 있다.

**LSTM.** 샘플을 앞에서부터 읽어 숨은 상태를 갱신한다. 마지막 숨은 상태를 선형층에 넣어 40샘플을 낸다.

**ModernTCN.** Luo & Wang, ICLR 2024. 합성곱만 쓴다. 이 셀의 구현은 논문의 재파라미터화와 다단계 다운샘플을 빼고 뼈대만 남겼다.

- RevIN: 심박마다 평균·분산을 맞춘 뒤 예측을 원래 스케일로 되돌린다.
- 패치: 길이 4, 보폭 2인 1차원 합성곱으로 100샘플을 토큰으로 줄인다.
- 큰 커널 깊이별 합성곱(kernel 13)으로 한 심박 안의 먼 지점을 보고 1×1 합성곱으로 채널을 섞는다.

아래는 가짜 배치 2개를 넣었을 때 나오는 모양이다.


In [ ]:
import inspect

from model import LSTMForecast, ModernTCN, show_shapes

print("LSTM")
show_shapes(LSTMForecast())
print()
print("ModernTCN")
show_shapes(ModernTCN())
print()
print(inspect.getsource(LSTMForecast.forward))
print(inspect.getsource(ModernTCN.forward))


## 7. 학습

손실은 L1(MAE)다. 검증 MAE가 가장 낮은 가중치를 `runs/lstm/best.pt`, `runs/moderntcn/best.pt`에 남긴다. 테스트는 아직 보지 않는다.

T4에서 두 모델의 학습은 수 분 안에 끝난다. zip에 이미 학습해 둔 파일이 있어도 이 셀을 실행하면 그 자리를 덮어 쓴다.


In [ ]:
import argparse

from train import set_seed, train_one

set_seed(42)
args = argparse.Namespace(
    model="all",
    data=ROOT / "dataset",
    runs=ROOT / "runs",
    epochs=300,
    batch=64,
    lr=1e-3,
    seed=42,
    workers=0,
)
for name in ("lstm", "moderntcn"):
    train_one(name, args, bundle, DEVICE)


## 8. 지표

테스트 전체 심박의 뒤 40샘플에 대해 아래를 계산한다. 숫자는 원래 진폭 스케일이다.

- MAE = |예측 − 정답| 의 평균.
- RMSE = 제곱 오차 평균의 제곱근. 큰 빗나감을 더 세게 본다.
- persist 나이브 = 입력의 마지막 값을 40칸 이어 붙인다.

진폭이 0 근처를 지나므로 MAPE는 쓰지 않는다. 네트워크가 persist보다 못하면 구조를 바꾼 보람이 없다.


In [ ]:
import json
from eval import metrics, naive_persist, predict_model

y = test["y"]
x = test["x"]
report = {
    "naive_persist": metrics(naive_persist(x), y),
}
for name in ("lstm", "moderntcn"):
    weights = ROOT / "runs" / name / "best.pt"
    pred = predict_model(name, weights, x, bundle["scaler"], DEVICE)
    report[name] = metrics(pred, y)
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "naive_persist": {
    "mae": 0.9462,
    "rmse": 1.364,
    "n": 4078,
    "horizon": 40
  },
  "lstm": {
    "mae": 0.4035,
    "rmse": 0.6634,
    "n": 4078,
    "horizon": 40
  },
  "moderntcn": {
    "mae": 0.3984,
    "rmse": 0.7281,
    "n": 4078,
    "horizon": 40
  }
}


## 9. 시연 네 심박

평가 숫자는 테스트 전체로 이미 냈다. 아래는 그 숫자를 대체하지 않는다. 실선이 입력이고 점선이 정답이며 색 선이 예측이다.


In [ ]:
import infer as infer_mod
from eval import naive_persist, predict_model

out_dir = ROOT / "runs" / "infer"
out_dir.mkdir(parents=True, exist_ok=True)
preds = {}
persist = naive_persist(test["x"])
for name in ("lstm", "moderntcn"):
    preds[name] = predict_model(name, ROOT / "runs" / name / "best.pt", test["x"], bundle["scaler"], DEVICE)
for ex in bundle["meta"]["examples"]:
    i = int(ex["split_index"])
    one = {name: preds[name][i, :, 0] for name in preds}
    title = f"{ex['id']}  {ex['title']}"
    path = out_dir / f"{ex['id']}_pred.png"
    infer_mod.draw_one(title, test["x"][i, :, 0], test["y"][i, :, 0], one, persist[i, :, 0], path)
    print(title)
    show_img(Image.open(path), width=980)
    print()
